# Neural Network Model

In [81]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

https://numpy.org/doc/2.4/reference/generated/numpy.array.html#numpy.array
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.index.html

## Load all feature files

In [82]:
# Helper function to clean numeric columns
def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

In [83]:
# State hospital features
state_hospitals = pd.read_csv("Features/State_Hospitals.csv", header=3)
state_hospitals.columns = state_hospitals.columns.str.strip()
state_hospitals = state_hospitals[state_hospitals["Provider State/Other"].notna()].copy()

hospital_cols = [
    "Provider State/Other",
    "Skilled Nursing Facility Beds Per 1,000 Part A Enrollees",
    "Short Stay Hospital Beds Per 1,000 Part A Enrollees",
    "Critical Access Hospital Beds Per 1,000 Part A Enrollees",
    "All Other Hospital Beds Per 1,000 Part A Enrollees"
]
state_hospitals = state_hospitals[hospital_cols].copy()

# State provider features
providers = pd.read_csv("Features/State_Providers.csv", header=2)
providers.columns = providers.columns.str.strip()
providers = providers[providers["Provider State/Other"].notna()].copy()

provider_cols = [
    "Provider State/Other",
    "Independent and Clinical Labs",
    "Ambulatory Surgical Centers",
    "Comprehensive Outpatient Rehabilitation Facilities",
    "Rural Health Clinics",
    "Federally Qualified Health Centers",
    "Hospices"
]
providers = providers[provider_cols].copy()

# State level payments
payments = pd.read_csv("Features/States_MedicarePaymentsandEnrollment.csv", header=3)
payments.columns = payments.columns.str.strip()
payments = payments[payments["Area of Residence"].notna()].copy()

payment_cols = [
    "Area of Residence",
    "Medicare Part A and/or Part B Program Payments Per Person With Utilization",
    "Medicare Part A and/or Part B Program Payments Per Original Medicare Enrollee"
]
payments = payments[payment_cols].copy()

# State level non institutional providers
noninst_providers = pd.read_csv("Features/States_NonInstitutional_Providers.csv", header=3)
noninst_providers.columns = noninst_providers.columns.str.strip()
noninst_providers = noninst_providers[noninst_providers["State/Other"].notna()].copy()

noninst_cols = ["State/Other", "2023"]
noninst_providers = noninst_providers[noninst_cols].copy()

print("Hospitals:", state_hospitals.shape)
print("Providers:", providers.shape)
print("Payments:", payments.shape)
print("Noninstitutional:", noninst_providers.shape)

Hospitals: (57, 5)
Providers: (57, 7)
Payments: (61, 3)
Noninstitutional: (61, 2)


## Clean and merge feature table

In [84]:
# Remove summary rows
bad_rows = ["All Areas", "United States"]

state_hospitals = state_hospitals[~state_hospitals["Provider State/Other"].isin(bad_rows)].copy()
providers = providers[~providers["Provider State/Other"].isin(bad_rows)].copy()
payments = payments[~payments["Area of Residence"].isin(bad_rows)].copy()
noninst_providers = noninst_providers[~noninst_providers["State/Other"].isin(bad_rows)].copy()

In [85]:
# Merge state level feature tables

feature_df = (
    state_hospitals
    .merge(providers, on="Provider State/Other", how="left")
    .merge(payments, left_on="Provider State/Other", right_on="Area of Residence", how="left")
    .merge(noninst_providers, left_on="Provider State/Other", right_on="State/Other", how="left")
    .drop(columns=["Area of Residence", "State/Other"])
    .rename(columns={
        "Provider State/Other": "State",
        "2023": "Noninstitutional Providers 2023"
    })
)

feature_df.head()
print("Feature df shape:", feature_df.shape)


Feature df shape: (55, 14)


# Load and prepare target values

In [86]:
inpatient_values = pd.read_csv("Predicted_values/MUP_INP_RY25_P03_V10_DY23_PrvSvc.csv")
inpatient_values.columns = inpatient_values.columns.str.strip()

inpatient_values = inpatient_values[inpatient_values["DRG_Desc"].str.contains("FEMALE")]

inpatient_values["DRG_Desc"].unique()

inpatient_drg_list = [
    "FEMALE REPRODUCTIVE SYSTEM RECONSTRUCTIVE PROCEDURES",
    "OTHER FEMALE REPRODUCTIVE SYSTEM O.R. PROCEDURES WITH CC/MCC",
    "MALIGNANCY, FEMALE REPRODUCTIVE SYSTEM WITH CC",
    "MALIGNANCY, FEMALE REPRODUCTIVE SYSTEM WITH MCC", 
    "D&C, CONIZATION, LAPAROSCOPY AND TUBAL INTERRUPTION WITH CC/MCC"
]

filtered_inpatient = inpatient_values[
    inpatient_values["DRG_Desc"].isin(inpatient_drg_list)
].copy()

filtered_inpatient["Tot_Dschrgs"] = pd.to_numeric(
    filtered_inpatient["Tot_Dschrgs"], errors="coerce"
)

state_inpatient_targets = (
    filtered_inpatient
    .groupby("Rndrng_Prvdr_State_Abrvtn", as_index=False)["Tot_Dschrgs"]
    .sum()
    .rename(columns={
        "Rndrng_Prvdr_State_Abrvtn": "State Abbrev",
        "Tot_Dschrgs": "Total Inpatient Discharges"
    })
)

state_inpatient_targets.head()

,State Abbrev,Total Inpatient Discharges
0,CA,13
1,CO,14
2,FL,57
3,IN,13
4,MA,40


In [87]:
outpatient_values = pd.read_csv("Predicted_values/MUP_OUT_RY25_P04_V10_DY23_Prov_Svc.csv")
outpatient_values.columns = outpatient_values.columns.str.strip()

# We included laparoscopy based off our research

target_apc_list = [
    "Level 1 Laparoscopy and Related Services",
    "Level 2 Laparoscopy and Related Services",
    "Level 4 Gynecologic Procedures",
    "Level 5 Gynecologic Procedures",
    "Level 6 Gynecologic Procedures"
]

filtered_outpatient = outpatient_values[
    outpatient_values["APC_Desc"].isin(target_apc_list)
].copy()

filtered_outpatient["Bene_Cnt"] = pd.to_numeric(
    filtered_outpatient["Bene_Cnt"], errors="coerce"
)

state_outpatient_targets = (
    filtered_outpatient
    .groupby("Rndrng_Prvdr_State_Abrvtn", as_index=False)["Bene_Cnt"]
    .sum()
    .rename(columns={
        "Rndrng_Prvdr_State_Abrvtn": "State Abbrev",
        "Bene_Cnt": "Total Outpatient Beneficiaries"
    })
)

state_outpatient_targets.head()

,State Abbrev,Total Outpatient Beneficiaries
0,AK,717.0
1,AL,4249.0
2,AR,3715.0
3,AZ,7443.0
4,CA,22800.0


In [88]:
state_targets = state_inpatient_targets.merge(
    state_outpatient_targets,
    on="State Abbrev",
    how="outer"
)

state_targets["Total Inpatient Discharges"] = state_targets["Total Inpatient Discharges"].fillna(0)
state_targets["Total Outpatient Beneficiaries"] = state_targets["Total Outpatient Beneficiaries"].fillna(0)

state_targets["Total Cases"] = (
    state_targets["Total Inpatient Discharges"] +
    state_targets["Total Outpatient Beneficiaries"]
)

print(state_targets.head())

  State Abbrev  Total Inpatient Discharges  Total Outpatient Beneficiaries  \
0           AK                         0.0                           717.0   
1           AL                         0.0                          4249.0   
2           AR                         0.0                          3715.0   
3           AZ                         0.0                          7443.0   
4           CA                        13.0                         22800.0   

   Total Cases  
0        717.0  
1       4249.0  
2       3715.0  
3       7443.0  
4      22813.0  


## Map state names to abbreviations

In [89]:
state_name_to_abbrev = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "District of Columbia": "DC", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI",
    "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME",
    "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
    "Mississippi": "MS", "Missouri": "MO", "Montana": "MT", "Nebraska": "NE",
    "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM",
    "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI",
    "South Carolina": "SC", "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX",
    "Utah": "UT", "Vermont": "VT", "Virginia": "VA", "Washington": "WA",
    "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY"
}

feature_df["State Abbrev"] = feature_df["State"].map(state_name_to_abbrev)

## Assemble model dataframe

In [90]:
model_df = feature_df.merge(state_targets, on="State Abbrev", how="inner")

print("Model df shape:", model_df.shape)
print(model_df[["State",
                "State Abbrev", 
                "Total Inpatient Discharges",
                "Total Outpatient Beneficiaries",
                "Total Cases"
]].head())

feature_columns = [
    "Skilled Nursing Facility Beds Per 1,000 Part A Enrollees",
    "Short Stay Hospital Beds Per 1,000 Part A Enrollees",
    "Critical Access Hospital Beds Per 1,000 Part A Enrollees",
    "All Other Hospital Beds Per 1,000 Part A Enrollees",
    "Medicare Part A and/or Part B Program Payments Per Person With Utilization",
    "Medicare Part A and/or Part B Program Payments Per Original Medicare Enrollee",
    "Noninstitutional Providers 2023"
]

Model df shape: (50, 18)
        State State Abbrev  Total Inpatient Discharges  \
0     Alabama           AL                         0.0   
1      Alaska           AK                         0.0   
2     Arizona           AZ                         0.0   
3    Arkansas           AR                         0.0   
4  California           CA                        13.0   

   Total Outpatient Beneficiaries  Total Cases  
0                          4249.0       4249.0  
1                           717.0        717.0  
2                          7443.0       7443.0  
3                          3715.0       3715.0  
4                         22800.0      22813.0  


## Clean model features

In [91]:
for col in feature_columns:
    model_df[col] = clean_numeric(model_df[col])

model_df["Total Cases"] = pd.to_numeric(model_df["Total Cases"], errors="coerce")

model_df = model_df.dropna(subset=["Total Cases"]).reset_index(drop=True)

for col in feature_columns:
    model_df[col] = model_df[col].fillna(model_df[col].median())

print("Model df shape:", model_df.shape)

Model df shape: (50, 18)


## Train test split

In [92]:
X_df = model_df[feature_columns].copy()
y_df = model_df["Total Cases"].copy()

n_total = len(model_df)
rng = np.random.default_rng(8)
all_idx = np.arange(n_total)
rng.shuffle(all_idx)

train_size = int(0.8 * n_total)
train_idx = all_idx[:train_size]
test_idx = all_idx[train_size:]

X_train_df = X_df.iloc[train_idx].copy()
X_test_df = X_df.iloc[test_idx].copy()
y_train_df = y_df.iloc[train_idx].copy()
y_test_df = y_df.iloc[test_idx].copy()

In [93]:
print(X_train_df)

    Skilled Nursing Facility Beds Per 1,000 Part A Enrollees  \
14                                               36.9          
16                                               30.3          
1                                                 7.3          
40                                               26.6          
0                                                24.1          
15                                               39.1          
11                                               13.9          
29                                               29.7          
39                                               16.7          
25                                               21.0          
5                                                18.8          
41                                               23.6          
13                                               32.3          
36                                               10.7          
12                                      

## Standardize predictors

In [94]:
train_means = X_train_df.mean()
train_stds = X_train_df.std().replace(0, 1)

X_train = ((X_train_df - train_means) / train_stds).to_numpy(dtype=float)
X_test = ((X_test_df - train_means) / train_stds).to_numpy(dtype=float)

y_train = y_train_df.to_numpy(dtype=float)
y_test = y_test_df.to_numpy(dtype=float)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Training points:", len(y_train))
print("Test points:", len(y_test))

Training shape: (40, 7)
Testing shape: (10, 7)
Training points: 40
Test points: 10


## Neural Network Definitions

In [95]:
eta = 0.001
q = 1
epochs = 300
lam = 0.01

def forward_pass_with_weights(input_x, W1, W2):
  h = np.maximum(0, W1.T @input_x)
  return W2.T@ h

def predict_matrix(Xmat, d, n):
    preds = []
    for i in range(Xmat.shape[0]):
        x = np.reshape(Xmat[i], (d, 1))
        preds.append(float(f(x)))
    return np.array(preds)

def f(x):
    h = np.maximum(0, W1.T.dot(x))
    return W2.T.dot(h)

errors = []
test_errors = []

#best_test_mse = float("inf")
#best_W1 = W1.copy()
#best_W2 = W2.copy()

def get_Ws(W1, W2, X, Y, d, n): 

    for epoch in range(epochs):
        dW2 = np.zeros_like(W2)
        dW1 = np.zeros_like(W1)

        for i in range(n):
            x = np.reshape(X[i], (d, 1))
            h = np.maximum(0, W1.T.dot(x))
            y_hat = W2.T.dot(h)

        dW2 += (2 / n) * (y_hat - y_train[i]) * h + 2 * lam * W2

        relu_grad = np.heaviside(h, 0)
        hidden_effect = W2 * relu_grad
        dW1 += (2 / n) * (y_hat - Y[i]) * (x.dot(hidden_effect.T)) + 2 * lam * W1

        W2 = W2 - eta * dW2
        W1 = W1 - eta * dW1

        train_preds = predict_matrix(X, d, n)
        #test_preds = predict_matrix(X_test, d, n)

        train_mse = np.mean((train_preds - Y) ** 2)
        #test_mse = np.mean((test_preds - y_test) ** 2)
        
        if epoch == 0: 
            best_test_mse = train_mse 
            best_W1 = W1.copy()
            best_W2 = W2.copy()

        # errors.append(train_mse)
        # test_errors.append(test_mse)

        if train_mse < best_test_mse:
            best_test_mse = train_mse
            best_W1 = W1.copy()
            best_W2 = W2.copy()

    W1 = best_W1
    W2 = best_W2
    
 
    return best_test_mse, W1, W2

## Cross validate

In [96]:

# Cross validate on training set for random ws and the number of hidden nodes 

choosing_parameters = np.zeros([30, 5])

# we used the leave one out method becasue we only have a few data points 

# we did cross validate to get the number of hidden nodes as well as the random seed to set w 

for number in range (0, 30): 
    np.random.seed(number)
    X_train_df.index = range(len(X_train_df))
    y_train_df.index = range(len(y_train_df))

    for element in range(1, 5):
        mse_iteration = []
        
        for df_row in range(len(X_train_df)):
            y_cv_test = np.array(y_train_df.iloc[df_row])
            x_cv_test = np.array(X_train_df.iloc[df_row])

            y_cv_train = y_train_df.drop(df_row, axis = 0).to_numpy()
            X = (X_train_df.drop(df_row, axis = 0)).to_numpy()

            d = X.shape[1]
            n = X.shape[0]

            W1 = np.random.randn(d, element) * 0.1
            W2 = np.random.randn(element, 1) * 0.1
            
            results = get_Ws(W1, W2, X, y_cv_train, d, n)
            W1_result = results[1]
            W2_result = results[2]
            
            
            pred_cv = forward_pass_with_weights(x_cv_test, W1_result, W2_result)
            cv_mse = np.mean((pred_cv - y_cv_test) ** 2)
            mse_iteration.append(cv_mse)
        
        choosing_parameters[number, element] = (sum(mse_iteration)/len(mse_iteration))
        
        

/var/folders/4j/3hhj_cmd3gnc44qd4552bf_40000gn/T/ipykernel_57500/3330296044.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  preds.append(float(f(x)))
/var/folders/4j/3hhj_cmd3gnc44qd4552bf_40000gn/T/ipykernel_57500/3330296044.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  preds.append(float(f(x)))
/var/folders/4j/3hhj_cmd3gnc44qd4552bf_40000gn/T/ipykernel_57500/3330296044.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  preds.append(float(f(x)))
/var

In [ ]:
for element in range(0, 5): 
    
    x = choosing_parameters[:, element]
    plt.plot(x)
    plt.show()

In [ ]:
# d 0 is null and we can not use that! 

print(choosing_parameters[0,1])

# using 1 hidden node with random seed 0!

## Train the neural network

In [95]:

d = X_test.shape[1]
np.random.seed(0)

# training the model 

W1 = np.random.randn(d, 1) * 0.1
W2 = np.random.randn(1, 1) * 0.1
results = get_Ws(W1, W2, X_train, y_train, d, n)

trained_W1 = results[1]

trained_W2 = results[2]

test_preds = []
#train_preds = predict_matrix(X_train)


for row in range(len(X_test)): 
    test_preds.append(forward_pass_with_weights(X_test[row], trained_W1, trained_W2))

#train_mae = np.mean(np.abs(train_preds - y_train))
test_mae = np.mean(np.abs(test_preds - y_test))

#train_rmse = np.sqrt(np.mean((train_preds - y_train) ** 2))
test_rmse = np.sqrt(np.mean((test_preds - y_test) ** 2))

#print(f"Train MAE:  {train_mae:.3f}")
print(f"Test MAE:   {test_mae:.3f}")
#print(f"Train RMSE: {train_rmse:.3f}")
print(f"Test RMSE:  {test_rmse:.3f}")

Test MAE:   60.505
Test RMSE:  63.884


## Visualize predictions 

Since the dataset is so small we figured it would be best just to print out the results we got comapred to the true values

In [96]:
print("PREDICTED: ")
print(test_preds[0])

print(test_preds[1])

print("ACTUAL: ")

print(y_test[0])
print(y_test[1])

PREDICTED: 
[0.]
[-0.01010255]
ACTUAL: 
81.0
40.0


In [97]:
print(W1)
print(W2)

[[ 0.17640523]
 [ 0.04001572]
 [ 0.0978738 ]
 [ 0.22408932]
 [ 0.1867558 ]
 [-0.09772779]
 [ 0.09500884]]
[[-0.01513572]]


Our predictions are really struggling to be pulled away from 0 telling us there is not really a relationship between the feature dataset and the the predicted column, or ideally under this model with these parameters, it is unable to accurately model the relationship.